# Problema 9.2 -- Produzione e manodopera: due formulazioni equivalenti

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/modellazione-mip/blob/main/notebooks/fam09_2_manodopera.ipynb)

La stessa decisione scritta due volte: con le *assunzioni* z_t (formulazione A)
oppure con l'*organico* y_t (formulazione B). Si dimostra che hanno lo stesso
insieme di piani ammissibili e lo stesso ottimo, e si confrontano i rilassamenti.
E' il tema del capitolo 4: due formulazioni si confrontano solo dopo aver
dimostrato che descrivono lo stesso insieme intero.

Il capitolo completo — modello, dati, risultati e analisi di sensitività — è [sul sito](https://fabiofurini.github.io/modellazione-mip/produzione-2/).

## Preparazione

La cella qui sotto installa `gurobipy` e scarica i tre moduli comuni del corso:
`stile.py` (palette), `mip.py` (rilassamento, duale, bound) ed `euristiche.py`
(next-fit, first-fit, best-fit). La licenza inclusa nel pacchetto pip è limitata a **2000
variabili e 2000 vincoli**: le istanze del corso sono piccole e ci stanno tutte
con ampio margine. Per istanze più grandi si attiva la licenza accademica
gratuita da [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Ambiente: il solver e i moduli comuni del corso.
# In locale usa il python/stile.py del repository; su Colab installa e scarica quello che manca.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

for modulo in ('stile', 'mip', 'euristiche'):                     # stile grafico e utilità del corso
    if importlib.util.find_spec(modulo) is None:
        locale = next((p for p in (Path(f"../python/{modulo}.py"), Path(f"python/{modulo}.py"))
                       if p.exists()), None)
        if locale is not None:
            sys.path.insert(0, str(locale.parent.resolve()))   # notebook aperto nel repository
        else:
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/fabiofurini/modellazione-mip/main/python/{modulo}.py", f"{modulo}.py")   # Colab

In [ ]:
import gurobipy as gp
import pandas as pd
from gurobipy import GRB

from mip import (ammissibile, due_rilassamenti, frazione, nuovo_modello, registra_bound,
                 rilassamento, risolvi, valuta)
from stile import ARANCIO, BLU, ROSSO, TEAL, intestazione, plt, salva_dati, salva_figura

R = range

# ---------- 1. MODELLO E ISTANZA ----------
intestazione("9.2 Produzione e manodopera: assunzioni (A) oppure organico (B)")
d2 = [60, 100, 140]        # domanda dei tre mesi (paia)
p2 = [15, 15, 15]          # costo di produzione per paio
h2 = [3, 3]                # costo di magazzino a fine mese
w2, r2, g2, u2, m2, r0 = 1500, 160, 4, 100, 2, 0
n2 = len(d2)
salva_dati(pd.DataFrame({"mese": R(1, n2 + 1), "domanda": d2, "costo_paio": p2}), "prod2_dati")
print(f"  {m2} operai all'inizio, {r2} h al mese ciascuno, {g2} h per paio: la capacita'")
print(f"  iniziale e' {m2 * r2 // g2} paia al mese. Salario {w2}, assunzione {u2}.")


def modello_A(d, p, h, w, r, g, u, m0, r0):
    """Formulazione A: z_t = quanti operai si assumono all'inizio del mese t."""
    n = len(d)
    mm = nuovo_modello("manodopera_A")
    x = mm.addVars(n, vtype=GRB.INTEGER, name="x")
    s = mm.addVars(n - 1, vtype=GRB.INTEGER, name="s")
    z = mm.addVars(n, vtype=GRB.INTEGER, name="z")
    mm.setObjective(gp.quicksum(p[t] * x[t] for t in R(n))
                    + gp.quicksum(h[t] * s[t] for t in R(n - 1))
                    + gp.quicksum((u + w * (n - t)) * z[t] for t in R(n)), GRB.MINIMIZE)
    mm.addConstr(x[0] - s[0] == d[0] - r0, name="bilancio[0]")
    mm.addConstrs((x[t] + s[t - 1] - s[t] == d[t] for t in R(1, n - 1)), name="bilancio")
    mm.addConstr(x[n - 1] + s[n - 2] == d[n - 1], name=f"bilancio[{n - 1}]")
    mm.addConstrs((-g * x[t] + gp.quicksum(r * z[j] for j in R(t + 1)) >= -r * m0
                   for t in R(n)), name="ore")
    return mm, x, s, z


def modello_B(d, p, h, w, r, g, u, m0, r0):
    """Formulazione B: y_t = quanti operai lavorano nel mese t (organico)."""
    n = len(d)
    mm = nuovo_modello("manodopera_B")
    x = mm.addVars(n, vtype=GRB.INTEGER, name="x")
    s = mm.addVars(n - 1, vtype=GRB.INTEGER, name="s")
    y = mm.addVars(n, vtype=GRB.INTEGER, name="y")
    # l'organico paga il salario ogni mese; le assunzioni sono gli incrementi y_t - y_{t-1}
    mm.setObjective(gp.quicksum(p[t] * x[t] for t in R(n))
                    + gp.quicksum(h[t] * s[t] for t in R(n - 1))
                    + gp.quicksum(w * y[t] for t in R(n))
                    + u * (y[n - 1] - m0), GRB.MINIMIZE)   # assunzioni totali = y_n - m0
    mm.addConstr(x[0] - s[0] == d[0] - r0, name="bilancio[0]")
    mm.addConstrs((x[t] + s[t - 1] - s[t] == d[t] for t in R(1, n - 1)), name="bilancio")
    mm.addConstr(x[n - 1] + s[n - 2] == d[n - 1], name=f"bilancio[{n - 1}]")
    mm.addConstrs((-g * x[t] + r * y[t] >= 0 for t in R(n)), name="ore")
    mm.addConstr(y[0] >= m0, name="organico_iniziale")
    mm.addConstrs((-y[t - 1] + y[t] >= 0 for t in R(1, n)), name="mai_licenziamenti")
    return mm, x, s, y


def duale_A(d, p, h, w, r, g, u, m0, r0):
    """max sum_t b_t mu_t - r m0 sum_t nu_t;  mu_t - g nu_t <= p_t;
    -mu_t + mu_{t+1} <= h_t;  r sum_{t >= j} nu_t <= u + w (n - j);  mu libere, nu >= 0."""
    n = len(d)
    b = [d[0] - r0] + d[1:n - 1] + [d[n - 1]]
    dl = nuovo_modello("duale_manodopera")
    mu = dl.addVars(n, lb=-GRB.INFINITY, name="mu")
    nu = dl.addVars(n, name="nu")
    dl.setObjective(gp.quicksum(b[t] * mu[t] for t in R(n))
                    - r * m0 * gp.quicksum(nu[t] for t in R(n)), GRB.MAXIMIZE)
    dl.addConstrs((mu[t] - g * nu[t] <= p[t] for t in R(n)), name="rc_x")
    dl.addConstrs((-mu[t] + mu[t + 1] <= h[t] for t in R(n - 1)), name="rc_s")
    dl.addConstrs((r * gp.quicksum(nu[t] for t in R(j, n)) <= u + w * (n - j) for j in R(n)),
                  name="rc_z")
    return dl


mA, xA, sA, zA = modello_A(d2, p2, h2, w2, r2, g2, u2, m2, r0)
mB, xB, sB, yB = modello_B(d2, p2, h2, w2, r2, g2, u2, m2, r0)
costante_A = m2 * w2 * n2          # il salario degli operai iniziali, fuori dal modello A
zA_val = risolvi(mA) + costante_A
zB_val = risolvi(mB)
print(f"  Formulazione A (assunzioni): z = {frazione(zA_val)} "
      f"(di cui {costante_A} di salario degli operai iniziali, termine costante)")
print(f"  Formulazione B (organico):   z = {frazione(zB_val)}")
assert abs(zA_val - zB_val) < 1e-6, (zA_val, zB_val)
print("  I due ottimi coincidono: le due formulazioni descrivono lo stesso problema.")
print("  Piano A: produzione " + ", ".join(frazione(xA[t].X) for t in R(n2))
      + "; assunzioni " + ", ".join(frazione(zA[t].X) for t in R(n2)))
print("  Piano B: produzione " + ", ".join(frazione(xB[t].X) for t in R(n2))
      + "; organico " + ", ".join(frazione(yB[t].X) for t in R(n2)))

# ---------- 2. L'EQUIVALENZA, VERIFICATA ----------
intestazione("9.2 L'equivalenza fra le due formulazioni, verificata")
print("  La corrispondenza e' y_t = m0 + sum_{j <= t} z_j, cioe' z_t = y_t - y_{t-1}")
print("  (con y_0 = m0). Sui piani ottimi:")
yA = [m2 + sum(round(zA[j].X) for j in R(t + 1)) for t in R(n2)]
print("    da A: organico implicito = " + ", ".join(str(v) for v in yA))
print("    da B: organico           = " + ", ".join(str(round(yB[t].X)) for t in R(n2)))
zB_implicite = [round(yB[0].X) - m2] + [round(yB[t].X) - round(yB[t - 1].X) for t in R(1, n2)]
print("    da B: assunzioni implicite = " + ", ".join(str(v) for v in zB_implicite))
assert sum(v * (u2 + w2 * (n2 - t)) for t, v in enumerate(zB_implicite)) + costante_A \
    == sum(round(zA[t].X) * (u2 + w2 * (n2 - t)) for t in R(n2)) + costante_A
print("  Il costo del personale coincide: A paga ogni assunzione una volta per tutti i mesi")
print("  che restano, B paga l'organico mese per mese. Stessa somma, contata in due modi.")

# ---------- 3. EURISTICA COSTRUTTIVA (UPPER BOUND) ----------
intestazione("9.2 Euristica, duale e bound")
# euristica costruttiva: si produce la domanda del mese, e si assume solo quando le ore non bastano
organico, assunzioni, prod = m2, [0] * n2, []
for t in R(n2):
    prod.append(d2[t])
    servono = -(-g2 * d2[t] // r2)               # ceil
    if organico < servono:
        assunzioni[t] = servono - organico
        organico = servono
    print(f"  Mese {t + 1}: si producono {d2[t]} paia, servono "
          f"ceil({g2} * {d2[t]} / {r2}) = {servono} operai; organico {organico - assunzioni[t]}"
          f" -> se ne assumono {assunzioni[t]}")
ub2 = sum(p2[t] * prod[t] for t in R(n2)) \
    + sum(assunzioni[t] * (u2 + w2 * (n2 - t)) for t in R(n2)) + costante_A
sol_eur = {f"x[{t}]": prod[t] for t in R(n2)} | {f"z[{t}]": assunzioni[t] for t in R(n2)} \
    | {f"s[{t}]": 0 for t in R(n2 - 1)}
assert ammissibile(mA, sol_eur)
print(f"  Costo dell'euristica: ub = {frazione(ub2)}")

# ---------- 4. DUALE E LOWER BOUND ----------
dl2 = duale_A(d2, p2, h2, w2, r2, g2, u2, m2, r0)
# ricetta: nu = 0 (le ore non si pagano) e mu_t = costo minimo per avere un paio al mese t
mu = []
for t in R(n2):
    mu.append(p2[t] if t == 0 else min(mu[t - 1] + h2[t - 1], p2[t]))
mano = {f"mu[{t}]": mu[t] for t in R(n2)}
lb2_var, viol = valuta(dl2, mano)
assert viol <= 1e-9, viol
lb2 = lb2_var + costante_A
print("  Duale a mano: nu = 0 (le ore di lavoro non si pagano) e mu_t = min(mu_{t-1}+h, p_t)")
print(f"    mu = " + ", ".join(frazione(v) for v in mu)
      + f"  ->  lb = {frazione(lb2_var)} + {costante_A} = {frazione(lb2)}")
zlp2, zlp2r, _ = due_rilassamenti(mA, dl2)
zlp2, zlp2r = zlp2 + costante_A, zlp2r + costante_A
riga = registra_bound("2 manodopera", ub2, lb2, zlp2, zlp2r, zA_val)
salva_dati(pd.DataFrame([riga]), "prod2_bound")
assert lb2 <= zlp2 <= zA_val <= ub2 + 1e-9

# ---------- 5. CONFRONTO DEI RILASSAMENTI DELLE DUE FORMULAZIONI ----------
zlpA, _, _ = rilassamento(mA, rafforzato=True)
zlpB, _, _ = rilassamento(mB, rafforzato=True)
print(f"  Rilassamenti: A -> {frazione(zlpA + costante_A)}   B -> {frazione(zlpB)}   "
      f"z(MILP) = {frazione(zA_val)}")
salva_dati(pd.DataFrame([{"formulazione": "A (assunzioni)", "z_lp": zlpA + costante_A,
                          "z_milp": zA_val},
                         {"formulazione": "B (organico)", "z_lp": zlpB, "z_milp": zB_val}]),
           "prod2_formulazioni")

# ---------- 6. DOMANDE DI MODELLAZIONE AGGIUNTIVE ----------
varianti = {}


def variante(nome, m, costante=0.0):
    z = risolvi(m) + costante
    print(f"  {nome:70s} z = {frazione(z)}")
    return z


# 2a: assumere costa molto di piu' (3000 invece di 100)
m, x, s, y = modello_B(d2, p2, h2, w2, r2, g2, 3000, m2, r0)
varianti["2a"] = variante("2a. L'assunzione costa 3000 euro invece di 100", m)
print("     organico: " + ", ".join(str(round(y[t].X)) for t in R(n2))
      + ";  produzione: " + ", ".join(str(round(x[t].X)) for t in R(n2)))
# 2b: straordinari, fino a 40 ore in piu' per operaio al mese, a 25 euro l'ora
m, x, s, y = modello_B(d2, p2, h2, w2, r2, g2, u2, m2, r0)
o = m.addVars(n2, name="o")
m.update()
for t in R(n2):
    m.chgCoeff(m.getConstrByName(f"ore[{t}]"), o[t], 1.0)   # le ore disponibili aumentano
m.addConstrs((o[t] <= 40 * y[t] for t in R(n2)), name="max_straordinari")
m.setObjective(m.getObjective() + gp.quicksum(25 * o[t] for t in R(n2)), GRB.MINIMIZE)
varianti["2b"] = variante("2b. Straordinari: fino a 40 h per operaio, 25 euro l'ora", m)
print("     straordinari usati: " + ", ".join(frazione(o[t].X) for t in R(n2))
      + "  (nessuno: anticipare la produzione e tenerla a magazzino costa meno)")
salva_dati(pd.DataFrame({"variante": list(varianti), "z": list(varianti.values())}),
           "prod2_varianti")

# ---------- 7. FIGURA ----------
fig, ax = plt.subplots(figsize=(7.0, 3.2))
mesi = list(R(1, n2 + 1))
ax.bar(mesi, [xB[t].X for t in R(n2)], color=TEAL, width=0.55, label="produzione $x_t$")
ax.plot(mesi, d2, "o--", color=ROSSO, label="domanda $d_t$")
ax2 = ax.twinx()
ax2.step(mesi, [yB[t].X for t in R(n2)], where="mid", color=BLU, lw=2, label="organico $y_t$")
ax2.set_ylabel("operai", color=BLU)
ax2.set_ylim(0, max(yB[t].X for t in R(n2)) + 1.5)
ax2.grid(False)
ax.set_xticks(mesi)
ax.set_xlabel("mese")
ax.set_ylabel("paia")
ax.set_title(f"9.2: piano ottimo (z = {frazione(zB_val)})")
ax.legend(fontsize=8, loc="upper left")
ax2.legend(fontsize=8, loc="lower right")
salva_figura(fig, "cap09_manodopera_ottimo")
print("Fine.")

---

Notebook generato da `python/fam09_2_manodopera.py` con `python3 python/genera_notebook.py`:
le modifiche si fanno sullo script, non qui.

Materiale didattico di [Fabio Furini](https://sites.google.com/view/fabiofurini/home-page) — DIAG, Sapienza Università di Roma.
Testi, figure e dati [CC BY 4.0](https://github.com/fabiofurini/modellazione-mip/blob/main/LICENSE),
codice [MIT](https://github.com/fabiofurini/modellazione-mip/blob/main/LICENSE-CODE).